In [ ]:
!pip uninstall -y google-adk opentelemetry-api opentelemetry-sdk \
    opentelemetry-exporter-otlp-proto-http opentelemetry-exporter-gcp-logging -q

In [ ]:
!pip install -q \
    chromadb \
    sentence-transformers \
    langchain-community \
    langchain-groq \
    langchain-google-genai \
    langgraph \
    streamlit \
    pyngrok

In [ ]:
import os, re, uuid, subprocess, time
from typing import TypedDict, List, Optional
from google.colab import userdata
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from langchain_groq import ChatGroq
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

In [ ]:
def _secret(key):
    try:
        return userdata.get(key)
    except Exception:
        return None

GROQ_API_KEY   = _secret("GROQ_API_KEY")
GOOGLE_API_KEY = _secret("GOOGLE_API_KEY")
HF_TOKEN       = _secret("HF_TOKEN")
NGROK_TOKEN    = _secret("NGROK_AUTH_TOKEN")

if GROQ_API_KEY:
    os.environ["GROQ_API_KEY"]   = GROQ_API_KEY
if GOOGLE_API_KEY:
    os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
if HF_TOKEN:
    os.environ["HUGGINGFACEHUB_API_TOKEN"] = HF_TOKEN

In [ ]:
def get_llm():
    if GROQ_API_KEY:
        return ChatGroq(
            model="llama-3.3-70b-versatile",
            api_key=GROQ_API_KEY,
            temperature=0.1,
        )
    if GOOGLE_API_KEY:
        from langchain_google_genai import ChatGoogleGenerativeAI
        return ChatGoogleGenerativeAI(
            model="gemini-1.5-flash",
            google_api_key=GOOGLE_API_KEY,
            temperature=0.1,
        )
    raise ValueError(
        "No LLM API key found. Add GROQ_API_KEY or GOOGLE_API_KEY to Colab secrets."
    )

In [ ]:
FINANCE_DOCS = [
    {"id": "doc_001", "topic": "Budgeting Basics",
     "text": "A budget is a plan for how you will spend and save your money each month. "
             "The 50/30/20 rule is a popular budgeting framework: 50% of after-tax income goes to needs "
             "(rent, groceries, utilities), 30% to wants (dining, entertainment, travel), and 20% to savings "
             "and debt repayment. Track every rupee using apps like Walnut or Money Manager. "
             "Review your budget monthly and adjust when income or expenses change."},
    {"id": "doc_002", "topic": "Emergency Fund",
     "text": "An emergency fund is money set aside exclusively for unexpected expenses like job loss, "
             "medical emergencies, or major repairs. Financial advisors recommend keeping 3 to 6 months of "
             "living expenses in a liquid, easily accessible account such as a savings account or liquid mutual fund. "
             "Start small — even Rs 1,000 a month builds a meaningful cushion over time. "
             "Never invest your emergency fund in stocks or long-term instruments."},
    {"id": "doc_003", "topic": "SIP and Mutual Funds",
     "text": "A Systematic Investment Plan (SIP) lets you invest a fixed amount regularly (monthly or weekly) "
             "into a mutual fund. SIPs benefit from rupee cost averaging — you buy more units when prices are low "
             "and fewer when they are high. Equity mutual funds are suitable for goals more than 5 years away. "
             "For goals 1-3 years away, consider debt or hybrid funds. Always check a fund's expense ratio "
             "and past performance across market cycles before investing."},
    {"id": "doc_004", "topic": "Tax Saving Investments",
     "text": "Section 80C of the Income Tax Act allows deductions up to Rs 1.5 lakh per year. "
             "Common 80C instruments include: PPF (Public Provident Fund) — 7.1% interest, 15-year lock-in; "
             "ELSS mutual funds — 3-year lock-in, equity growth potential; NSC (National Savings Certificate); "
             "5-year tax-saving FD; and life insurance premiums. NPS (National Pension System) gives an "
             "additional Rs 50,000 deduction under Section 80CCD(1B). Choose instruments matching your liquidity needs."},
    {"id": "doc_005", "topic": "Credit Score and Loans",
     "text": "A credit score (CIBIL score) ranges from 300 to 900. Scores above 750 are considered excellent "
             "and qualify you for lower interest rates on home loans, car loans, and personal loans. "
             "Improve your score by: paying EMIs and credit card bills on time, keeping credit utilisation "
             "below 30%, not applying for multiple loans simultaneously, and maintaining a mix of secured "
             "and unsecured loans. Check your free credit report annually at CIBIL, Experian, or CRIF."},
    {"id": "doc_006", "topic": "Home Loan Guidance",
     "text": "A home loan (housing loan) in India typically covers up to 80% of the property value. "
             "Key terms: Principal — the borrowed amount; EMI — Equated Monthly Instalment covering principal "
             "and interest; Tenure — usually 15-30 years. Compare interest rates: SBI, HDFC, ICICI offer "
             "competitive rates linked to RBI repo rate. Tax benefit: Principal repayment qualifies under 80C; "
             "interest paid qualifies under Section 24(b) up to Rs 2 lakh. Always prepay when you have surplus cash."},
    {"id": "doc_007", "topic": "Insurance Basics",
     "text": "Insurance protects your finances against large unexpected losses. Key types: "
             "Term Life Insurance — provides a large cover (e.g., Rs 1 crore) at low premium; buy 10-15x your "
             "annual income. Health Insurance — covers hospitalisation; buy a family floater of at least Rs 5 lakh. "
             "Vehicle Insurance — third-party is mandatory by law in India. "
             "Avoid insurance-cum-investment products like ULIPs and endowment plans — they offer poor returns. "
             "Buy pure term + separate mutual fund investments instead."},
    {"id": "doc_008", "topic": "Stock Market Basics",
     "text": "The stock market allows you to buy ownership (equity) in companies. In India, stocks are traded "
             "on BSE (Bombay Stock Exchange) and NSE (National Stock Exchange). A Demat account is required to "
             "hold shares electronically. Key concepts: P/E ratio measures valuation; dividend is a share of profits "
             "paid to investors; blue-chip stocks are shares of large, stable companies. "
             "Never invest money you cannot afford to lose. Diversify across sectors. "
             "Long-term equity investing (5+ years) historically beats inflation in India."},
    {"id": "doc_009", "topic": "Retirement Planning",
     "text": "Start retirement planning as early as possible — the power of compounding grows wealth exponentially. "
             "Estimate your retirement corpus: monthly expenses x 12 x 25 (using the 4% withdrawal rule). "
             "Key instruments: EPF (Employees Provident Fund) — mandatory for salaried employees; "
             "NPS (National Pension System) — low cost, equity + debt mix; "
             "PPF — safe, government-backed 15-year product. "
             "At 30, aim to save at least 15% of income for retirement. At 40, increase to 25%."},
    {"id": "doc_010", "topic": "Debt Management",
     "text": "High-interest debt like credit card debt (36-42% per annum) and personal loans (12-24%) "
             "should be paid off as fast as possible. Use the avalanche method: list debts by interest rate, "
             "pay minimum on all, and throw every extra rupee at the highest-rate debt first. "
             "Debt consolidation loans can simplify multiple debts into one lower-rate loan. "
             "Avoid taking new loans to repay old ones. Build an emergency fund before aggressively investing, "
             "so you never have to rely on credit cards for unexpected expenses."},
    {"id": "doc_011", "topic": "Gold and Real Estate",
     "text": "Gold has been a traditional store of value in India. Sovereign Gold Bonds (SGBs) issued by RBI "
             "are the best way to invest in gold — they pay 2.5% annual interest and have no making charges. "
             "Avoid physical gold jewellery as an investment due to high making charges and impurity risk. "
             "Real estate can be a good long-term asset but requires large capital, is illiquid, and involves "
             "transaction costs (registration, stamp duty). REITs (Real Estate Investment Trusts) allow "
             "investment in commercial real estate with small amounts and provide regular dividends."},
    {"id": "doc_012", "topic": "Financial Goal Setting",
     "text": "Effective financial planning starts with SMART goals: Specific, Measurable, Achievable, "
             "Relevant, Time-bound. Categorise goals: Short-term (under 1 year) — vacation, gadget purchase; "
             "Medium-term (1-5 years) — car, higher education; Long-term (5+ years) — house, retirement. "
             "Match investment instruments to goal horizon: short-term goals in FD/liquid funds, "
             "medium-term in hybrid funds, long-term in equity funds. "
             "Review and update your financial plan every year or after major life events."},
]

In [ ]:
LANGUAGE_INSTRUCTIONS = {
    "en": (
        "You MUST respond ONLY in English. "
        "Do NOT use Hindi, Bengali, or any other language."
    ),
    "hi": (
        "You MUST respond ONLY in Hindi using the Devanagari script (e.g. \u0906\u092a, \u0915\u093e, \u0939\u0948). "
        "Do NOT use Roman/Latin letters to write Hindi (no aap, ka, hai). "
        "Do NOT transliterate. Do NOT use English sentences. "
        "Technical terms like SIP, EMI, PPF may be kept as-is in Devanagari context."
    ),
    "bn": (
        "You MUST respond ONLY in Bengali using the Bengali script (e.g. \u0986\u09aa\u09a8\u09bf, \u098f\u099f\u09bf, \u09b9\u09af\u09bc). "
        "Do NOT use Roman/Latin letters to write Bengali. "
        "Do NOT transliterate. Do NOT use English sentences. "
        "Technical terms like SIP, EMI, PPF may be kept as-is in Bengali context."
    ),
}

LANGUAGE_NAMES = {"en": "English", "hi": "Hindi", "bn": "Bengali"}

In [ ]:
print("Loading embedding model...")
embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

texts     = [doc["text"]             for doc in FINANCE_DOCS]
metadatas = [{"topic": doc["topic"]} for doc in FINANCE_DOCS]

vectordb = Chroma.from_texts(texts=texts, embedding=embedding_model, metadatas=metadatas)
print(f"ChromaDB loaded with {len(texts)} documents.\n")

# Retrieval test — must pass before building the graph
_test = vectordb.similarity_search("how to start investing in mutual funds", k=2)
print("Retrieval Test ::")
for r in _test:
    print(f"  Topic matched: {r.metadata['topic']}")
print("Retrieval OK.\n")

def search_kb(query: str, top_k: int = 3) -> str:
    results = vectordb.similarity_search(query, k=top_k)
    return "\n\n".join(f"[{r.metadata['topic']}]\n{r.page_content}" for r in results)

In [ ]:
def emi_tool(principal: float, annual_rate: float, tenure_months: int) -> str:
    try:
        if any(v <= 0 for v in [principal, annual_rate, tenure_months]):
            return "All values (principal, rate, tenure) must be positive numbers."
        r   = annual_rate / (12 * 100)
        emi = (principal * r * (1 + r) ** tenure_months) / ((1 + r) ** tenure_months - 1)
        total = emi * tenure_months
        return (
            f"Principal  : Rs {principal:,.0f}\n"
            f"Rate       : {annual_rate}% per annum\n"
            f"Tenure     : {tenure_months} months ({tenure_months//12} years)\n"
            f"Monthly EMI: Rs {emi:,.2f}\n"
            f"Total Paid : Rs {total:,.2f}\n"
            f"Interest   : Rs {total - principal:,.2f}"
        )
    except Exception as e:
        return f"Calculation error: {e}"

In [ ]:
class FinState(TypedDict):
    question:     str
    messages:     List[dict]
    route:        str
    context:      str
    answer:       str
    language:     str
    faithfulness: float
    eval_retries: int
    user_name:    Optional[str]

In [ ]:
def memory_node(state: FinState) -> FinState:
    msgs = state.get("messages", [])
    q    = state["question"]
    name = state.get("user_name")
    for phrase in ["my name is", "i am ", "i'm "]:
        if phrase in q.lower():
            after = q.lower().split(phrase, 1)[1].strip()
            candidate = after.split()[0].capitalize() if after else None
            if candidate and candidate.isalpha():
                name = candidate
    msgs = msgs + [{"role": "user", "content": q}]
    msgs = msgs[-10:]  # sliding window
    return {**state, "messages": msgs, "user_name": name, "eval_retries": 0}

In [ ]:
def router_node(state: FinState) -> FinState:
    llm = get_llm()
    history = "\n".join(f"{m['role']}: {m['content']}" for m in state["messages"][-4:])
    prompt = f"""You are a router for a personal finance chatbot. Read the user's question and choose ONE route.

Routes:
- retrieve  : any question about personal finance topics — budgeting, saving, investing, SIP,
              mutual funds, stocks, loans, EMI concepts, credit score, insurance, tax, retirement,
              debt, gold, REIT, PPF, NPS, ELSS, emergency fund, goal setting
- tool      : user wants to CALCULATE a loan EMI and has provided numbers (principal, rate, tenure)
- chitchat  : greeting, thank you, casual talk, completely off-topic (health, weather, etc.)

Recent history:
{history}

User question: {state["question"]}

Reply with ONLY ONE word — retrieve, tool, or chitchat."""

    raw = llm.invoke(prompt).content.strip().lower()
    if "tool" in raw:
        route = "tool"
    elif "chitchat" in raw:
        route = "chitchat"
    else:
        route = "retrieve"

    print(f"[router] route={route}")
    return {**state, "route": route}

In [ ]:
def retrieval_node(state: FinState) -> FinState:
    context = search_kb(state["question"])
    print(f"[retrieval] {len(context)} chars fetched")
    return {**state, "context": context}


def skip_node(state: FinState) -> FinState:
    return {**state, "context": ""}

In [ ]:
def tool_node(state: FinState) -> FinState:
    q    = state["question"]
    nums = list(map(float, re.findall(r"\d+(?:\.\d+)?", q)))

    if len(nums) >= 3:
        raw_result = emi_tool(nums[0], nums[1], int(nums[2]))
    elif len(nums) == 2:
        raw_result = (
            "I need three values to calculate EMI: principal amount, "
            "annual interest rate, and tenure in months. "
            "Example: 'EMI for 500000 at 8.5% for 240 months'"
        )
    else:
        raw_result = (
            "To calculate your loan EMI, please provide the principal amount, "
            "annual interest rate, and tenure in months. "
            "Example: 'EMI for 500000 at 8.5% for 240 months'"
        )

    return {**state, "context": "", "answer": raw_result}

In [ ]:
def answer_node(state: FinState) -> FinState:
    llm  = get_llm()
    lang = state.get("language", "en")
    lang_instruction = LANGUAGE_INSTRUCTIONS[lang]

    name_note  = f"The user's name is {state['user_name']}. " if state.get("user_name") else ""
    history    = "\n".join(f"{m['role']}: {m['content']}" for m in state["messages"][-6:])
    context    = state.get("context", "")
    raw_ans    = state.get("answer", "")
    retry_note = " IMPORTANT: Your previous answer was not faithful to the context — be more precise." if state.get("eval_retries", 0) > 0 else ""

    if raw_ans and not context:
        prompt = f"""{lang_instruction}\n\n{name_note}Present the following loan EMI calculation result to the user in a clear, friendly way.\nTranslate all labels and explanatory text into the required language.\nKeep the numbers exactly as they are.\n\nResult to present:\n{raw_ans}\n\nREMINDER: {lang_instruction}"""

    elif context:
        prompt = f"""{lang_instruction}\n\n{name_note}You are FinAdvisor, a personal finance assistant for users.{retry_note}\n\nRULES:\n- Answer ONLY using the context below.\n- If the answer is not in the context, say so clearly in the required language.\n- Do NOT mix languages.\n\nContext:\n{context}\n\nConversation history:\n{history}\n\nQuestion: {state["question"]}\n\nREMINDER: {lang_instruction}"""

    else:
        prompt = f"""{lang_instruction}\n\n{name_note}You are FinAdvisor, a helpful personal finance assistant for users.\nRespond warmly and briefly. If the user asks a finance question, let them know you can help with\nbudgeting, SIP, tax saving, loans, insurance, stocks, retirement, and debt management.\n\nQuestion: {state["question"]}\n\nREMINDER: {lang_instruction}"""

    answer = llm.invoke(prompt).content.strip()
    return {**state, "answer": answer}

In [ ]:
def eval_node(state: FinState) -> FinState:
    if not state.get("context"):
        return {**state, "faithfulness": 1.0}

    llm = get_llm()
    prompt = f"""Score the faithfulness of the answer to the context on a scale of 0.0 to 1.0.\nFaithfulness = does the answer use ONLY information present in the context?\n1.0 = fully faithful. 0.0 = completely made up.\n\nContext:\n{state["context"][:800]}\n\nAnswer:\n{state["answer"][:500]}\n\nReply with ONLY a decimal number between 0.0 and 1.0."""

    try:
        score = float(llm.invoke(prompt).content.strip())
        score = max(0.0, min(1.0, score))
    except Exception:
        score = 0.8

    retries = state.get("eval_retries", 0) + 1
    print(f"[eval] faithfulness={score:.2f}  retries={retries}")
    return {**state, "faithfulness": score, "eval_retries": retries}

In [ ]:
def save_node(state: FinState) -> FinState:
    msgs = state.get("messages", [])
    msgs = (msgs + [{"role": "assistant", "content": state["answer"]}])[-12:]
    return {**state, "messages": msgs}


def route_decision(state: FinState) -> str:
    return state["route"]


def eval_decision(state: FinState) -> str:
    if state.get("faithfulness", 1.0) < 0.7 and state.get("eval_retries", 0) < 2:
        return "answer"
    return "save"

In [ ]:
def build_graph():
    memory = MemorySaver()
    g = StateGraph(FinState)

    g.add_node("memory",    memory_node)
    g.add_node("router",    router_node)
    g.add_node("retrieval", retrieval_node)
    g.add_node("skip",      skip_node)
    g.add_node("tool",      tool_node)
    g.add_node("answer",    answer_node)
    g.add_node("eval",      eval_node)
    g.add_node("save",      save_node)

    g.set_entry_point("memory")
    g.add_edge("memory", "router")

    g.add_conditional_edges("router", route_decision, {
        "retrieve": "retrieval",
        "chitchat": "skip",
        "tool":     "tool",
    })

    g.add_edge("retrieval", "answer")
    g.add_edge("skip",      "answer")
    g.add_edge("tool",      "answer")
    g.add_edge("answer",    "eval")

    g.add_conditional_edges("eval", eval_decision, {
        "answer": "answer",
        "save":   "save",
    })

    g.add_edge("save", END)
    return g.compile(checkpointer=memory)


app_graph = build_graph()
print("Graph compiled successfully.\n")

In [ ]:
def _blank_state(language="en"):
    return {
        "question":     "",
        "messages":     [],
        "route":        "",
        "context":      "",
        "answer":       "",
        "language":     language,
        "faithfulness": 1.0,
        "eval_retries": 0,
        "user_name":    None,
    }


def run(question: str, language: str = "en", thread_id: str = "default"):
    state = _blank_state(language)
    state["question"] = question
    config = {"configurable": {"thread_id": thread_id}}
    result = app_graph.invoke(state, config)
    print(f"\nLang={language} | Q: {question}")
    print(f"A: {result['answer']}")
    print("─" * 60)
    return result

In [ ]:
# 1. Finance RAG — English
run("What is SIP and how does it work?",          language="en", thread_id="en1")

In [ ]:
# 2. Finance RAG — Hindi (must use Devanagari, no romanised Hindi)
run("SIP \u0915\u094d\u092f\u093e \u0939\u0948?", language="hi", thread_id="hi1")

In [ ]:
# 3. Finance RAG — Bengali (must use Bengali script)
run("SIP \u0995\u09c0 \u098f\u09ac\u0982 \u098f\u099f\u09bf \u0995\u09c0\u09ad\u09be\u09ac\u09c7 \u0995\u09be\u099c \u0995\u09b0\u09c7?", language="bn", thread_id="bn1")

In [ ]:
# 4. EMI tool — English
run("Calculate EMI for 500000 at 8.5 for 240 months", language="en", thread_id="emi_en")

In [ ]:
# 5. EMI tool — Hindi
run("500000 \u0915\u093e 8.5% \u092a\u0930 240 \u092e\u0939\u0940\u0928\u0947 \u0915\u093e EMI calculate \u0915\u0930\u094b", language="hi", thread_id="emi_hi")

In [ ]:
# 6. EMI tool — Bengali
run("500000 \u099f\u09be\u0995\u09be\u09b0 EMI calculate \u0995\u09b0\u09c1\u09a8 8.5% \u098f 240 \u09ae\u09be\u09b8\u09c7\u09b0 \u099c\u09a8\u09cd\u09af", language="bn", thread_id="emi_bn")

In [ ]:
# 7. Multi-turn memory test (same thread_id)
run("What is an emergency fund?",   language="en", thread_id="mem1")
run("How much should I keep in it?", language="en", thread_id="mem1")

In [ ]:
# 8. Chitchat
run("Hello! How are you?", language="en", thread_id="chat1")

In [ ]:
# 9. Out-of-scope
run("What is the cure for diabetes?", language="en", thread_id="oos1")

In [ ]:
APP_CODE = r'''
import os, uuid, re
import streamlit as st
from typing import TypedDict, List, Optional
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

FINANCE_DOCS = [
    {"id": "doc_001", "topic": "Budgeting Basics",
     "text": "A budget is a plan for how you will spend and save your money each month. "
             "The 50/30/20 rule is a popular budgeting framework: 50% of after-tax income goes to needs "
             "(rent, groceries, utilities), 30% to wants (dining, entertainment, travel), and 20% to savings "
             "and debt repayment. Track every rupee using apps like Walnut or Money Manager. "
             "Review your budget monthly and adjust when income or expenses change."},
    {"id": "doc_002", "topic": "Emergency Fund",
     "text": "An emergency fund is money set aside exclusively for unexpected expenses like job loss, "
             "medical emergencies, or major repairs. Financial advisors recommend keeping 3 to 6 months of "
             "living expenses in a liquid, easily accessible account such as a savings account or liquid mutual fund. "
             "Start small — even Rs 1,000 a month builds a meaningful cushion over time. "
             "Never invest your emergency fund in stocks or long-term instruments."},
    {"id": "doc_003", "topic": "SIP and Mutual Funds",
     "text": "A Systematic Investment Plan (SIP) lets you invest a fixed amount regularly (monthly or weekly) "
             "into a mutual fund. SIPs benefit from rupee cost averaging — you buy more units when prices are low "
             "and fewer when they are high. Equity mutual funds are suitable for goals more than 5 years away. "
             "For goals 1-3 years away, consider debt or hybrid funds. Always check a fund's expense ratio "
             "and past performance across market cycles before investing."},
    {"id": "doc_004", "topic": "Tax Saving Investments",
     "text": "Section 80C of the Income Tax Act allows deductions up to Rs 1.5 lakh per year. "
             "Common 80C instruments include: PPF (Public Provident Fund) — 7.1% interest, 15-year lock-in; "
             "ELSS mutual funds — 3-year lock-in, equity growth potential; NSC (National Savings Certificate); "
             "5-year tax-saving FD; and life insurance premiums. NPS (National Pension System) gives an "
             "additional Rs 50,000 deduction under Section 80CCD(1B). Choose instruments matching your liquidity needs."},
    {"id": "doc_005", "topic": "Credit Score and Loans",
     "text": "A credit score (CIBIL score) ranges from 300 to 900. Scores above 750 are considered excellent "
             "and qualify you for lower interest rates on home loans, car loans, and personal loans. "
             "Improve your score by: paying EMIs and credit card bills on time, keeping credit utilisation "
             "below 30%, not applying for multiple loans simultaneously, and maintaining a mix of secured "
             "and unsecured loans. Check your free credit report annually at CIBIL, Experian, or CRIF."},
    {"id": "doc_006", "topic": "Home Loan Guidance",
     "text": "A home loan (housing loan) in India typically covers up to 80% of the property value. "
             "Key terms: Principal — the borrowed amount; EMI — Equated Monthly Instalment covering principal "
             "and interest; Tenure — usually 15-30 years. Compare interest rates: SBI, HDFC, ICICI offer "
             "competitive rates linked to RBI repo rate. Tax benefit: Principal repayment qualifies under 80C; "
             "interest paid qualifies under Section 24(b) up to Rs 2 lakh. Always prepay when you have surplus cash."},
    {"id": "doc_007", "topic": "Insurance Basics",
     "text": "Insurance protects your finances against large unexpected losses. Key types: "
             "Term Life Insurance — provides a large cover (e.g., Rs 1 crore) at low premium; buy 10-15x your "
             "annual income. Health Insurance — covers hospitalisation; buy a family floater of at least Rs 5 lakh. "
             "Vehicle Insurance — third-party is mandatory by law in India. "
             "Avoid insurance-cum-investment products like ULIPs and endowment plans — they offer poor returns. "
             "Buy pure term + separate mutual fund investments instead."},
    {"id": "doc_008", "topic": "Stock Market Basics",
     "text": "The stock market allows you to buy ownership (equity) in companies. In India, stocks are traded "
             "on BSE (Bombay Stock Exchange) and NSE (National Stock Exchange). A Demat account is required to "
             "hold shares electronically. Key concepts: P/E ratio measures valuation; dividend is a share of profits "
             "paid to investors; blue-chip stocks are shares of large, stable companies. "
             "Never invest money you cannot afford to lose. Diversify across sectors. "
             "Long-term equity investing (5+ years) historically beats inflation in India."},
    {"id": "doc_009", "topic": "Retirement Planning",
     "text": "Start retirement planning as early as possible — the power of compounding grows wealth exponentially. "
             "Estimate your retirement corpus: monthly expenses x 12 x 25 (using the 4% withdrawal rule). "
             "Key instruments: EPF (Employees Provident Fund) — mandatory for salaried employees; "
             "NPS (National Pension System) — low cost, equity + debt mix; "
             "PPF — safe, government-backed 15-year product. "
             "At 30, aim to save at least 15% of income for retirement. At 40, increase to 25%."},
    {"id": "doc_010", "topic": "Debt Management",
     "text": "High-interest debt like credit card debt (36-42% per annum) and personal loans (12-24%) "
             "should be paid off as fast as possible. Use the avalanche method: list debts by interest rate, "
             "pay minimum on all, and throw every extra rupee at the highest-rate debt first. "
             "Debt consolidation loans can simplify multiple debts into one lower-rate loan. "
             "Avoid taking new loans to repay old ones. Build an emergency fund before aggressively investing, "
             "so you never have to rely on credit cards for unexpected expenses."},
    {"id": "doc_011", "topic": "Gold and Real Estate",
     "text": "Gold has been a traditional store of value in India. Sovereign Gold Bonds (SGBs) issued by RBI "
             "are the best way to invest in gold — they pay 2.5% annual interest and have no making charges. "
             "Avoid physical gold jewellery as an investment due to high making charges and impurity risk. "
             "Real estate can be a good long-term asset but requires large capital, is illiquid, and involves "
             "transaction costs (registration, stamp duty). REITs (Real Estate Investment Trusts) allow "
             "investment in commercial real estate with small amounts and provide regular dividends."},
    {"id": "doc_012", "topic": "Financial Goal Setting",
     "text": "Effective financial planning starts with SMART goals: Specific, Measurable, Achievable, "
             "Relevant, Time-bound. Categorise goals: Short-term (under 1 year) — vacation, gadget purchase; "
             "Medium-term (1-5 years) — car, higher education; Long-term (5+ years) — house, retirement. "
             "Match investment instruments to goal horizon: short-term goals in FD/liquid funds, "
             "medium-term in hybrid funds, long-term in equity funds. "
             "Review and update your financial plan every year or after major life events."},
]

LANGUAGE_INSTRUCTIONS = {
    "en": (
        "You MUST respond ONLY in English. "
        "Do NOT use Hindi, Bengali, or any other language."
    ),
    "hi": (
        "You MUST respond ONLY in Hindi using the Devanagari script (e.g. \u0906\u092a, \u0915\u093e, \u0939\u0948). "
        "Do NOT use Roman/Latin letters to write Hindi (no aap, ka, hai). "
        "Do NOT transliterate. Do NOT use English sentences. "
        "Technical terms like SIP, EMI, PPF may be kept as-is in Devanagari context."
    ),
    "bn": (
        "You MUST respond ONLY in Bengali using the Bengali script (e.g. \u0986\u09aa\u09a8\u09bf, \u098f\u099f\u09bf, \u09b9\u09af\u09bc). "
        "Do NOT use Roman/Latin letters to write Bengali. "
        "Do NOT transliterate. Do NOT use English sentences. "
        "Technical terms like SIP, EMI, PPF may be kept as-is in Bengali context."
    ),
}

def get_llm():
    groq_key   = os.environ.get("GROQ_API_KEY")
    google_key = os.environ.get("GOOGLE_API_KEY")
    if groq_key:
        from langchain_groq import ChatGroq
        return ChatGroq(model="llama-3.3-70b-versatile", api_key=groq_key, temperature=0.1)
    if google_key:
        from langchain_google_genai import ChatGoogleGenerativeAI
        return ChatGoogleGenerativeAI(model="gemini-1.5-flash", google_api_key=google_key, temperature=0.1)
    raise ValueError("No API key found. Set GROQ_API_KEY or GOOGLE_API_KEY in environment.")

@st.cache_resource(show_spinner="Loading knowledge base...")
def load_vectordb():
    embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
    texts     = [doc["text"]             for doc in FINANCE_DOCS]
    metadatas = [{"topic": doc["topic"]} for doc in FINANCE_DOCS]
    return Chroma.from_texts(texts=texts, embedding=embedding_model, metadatas=metadatas)

def search_kb(vectordb, query: str, top_k: int = 3) -> str:
    results = vectordb.similarity_search(query, k=top_k)
    return "\n\n".join(f"[{r.metadata['topic']}]\n{r.page_content}" for r in results)

def emi_tool(principal: float, annual_rate: float, tenure_months: int) -> str:
    try:
        if any(v <= 0 for v in [principal, annual_rate, tenure_months]):
            return "All values (principal, rate, tenure) must be positive numbers."
        r     = annual_rate / (12 * 100)
        emi   = (principal * r * (1 + r) ** tenure_months) / ((1 + r) ** tenure_months - 1)
        total = emi * tenure_months
        return (
            f"Principal  : Rs {principal:,.0f}\n"
            f"Rate       : {annual_rate}% per annum\n"
            f"Tenure     : {tenure_months} months ({tenure_months // 12} years)\n"
            f"Monthly EMI: Rs {emi:,.2f}\n"
            f"Total Paid : Rs {total:,.2f}\n"
            f"Interest   : Rs {total - principal:,.2f}"
        )
    except Exception as e:
        return f"Calculation error: {e}"

class FinState(TypedDict):
    question:     str
    messages:     List[dict]
    route:        str
    context:      str
    answer:       str
    language:     str
    faithfulness: float
    eval_retries: int
    user_name:    Optional[str]

def memory_node(state: FinState) -> FinState:
    msgs = state.get("messages", [])
    q    = state["question"]
    name = state.get("user_name")
    for phrase in ["my name is", "i am ", "i'm "]:
        if phrase in q.lower():
            after = q.lower().split(phrase, 1)[1].strip()
            candidate = after.split()[0].capitalize() if after else None
            if candidate and candidate.isalpha():
                name = candidate
    msgs = (msgs + [{"role": "user", "content": q}])[-10:]
    return {**state, "messages": msgs, "user_name": name, "eval_retries": 0}

def router_node(state: FinState) -> FinState:
    llm     = get_llm()
    history = "\n".join(f"{m['role']}: {m['content']}" for m in state["messages"][-4:])
    prompt  = (
        "You are a router for a personal finance chatbot. Read the user question and choose ONE route.\n\n"
        "Routes:\n"
        "- retrieve  : any question about personal finance — budgeting, saving, investing, SIP, mutual funds,\n"
        "              stocks, loans, EMI concepts, credit score, insurance, tax, retirement, debt, gold,\n"
        "              REIT, PPF, NPS, ELSS, emergency fund, goal setting\n"
        "- tool      : user wants to CALCULATE a loan EMI and has provided numbers (principal, rate, tenure)\n"
        "- chitchat  : greeting, thank you, casual talk, completely off-topic\n\n"
        f"Recent history:\n{history}\n\n"
        f"User question: {state['question']}\n\n"
        "Reply with ONLY ONE word — retrieve, tool, or chitchat."
    )
    raw   = llm.invoke(prompt).content.strip().lower()
    route = "tool" if "tool" in raw else "chitchat" if "chitchat" in raw else "retrieve"
    return {**state, "route": route}

def retrieval_node(state: FinState) -> FinState:
    vectordb = load_vectordb()
    context  = search_kb(vectordb, state["question"])
    return {**state, "context": context}

def skip_node(state: FinState) -> FinState:
    return {**state, "context": ""}

def tool_node(state: FinState) -> FinState:
    q    = state["question"]
    nums = list(map(float, re.findall(r"\d+(?:\.\d+)?", q)))
    if len(nums) >= 3:
        raw_result = emi_tool(nums[0], nums[1], int(nums[2]))
    elif len(nums) == 2:
        raw_result = "I need three values: principal, annual interest rate, and tenure in months. Example: EMI for 500000 at 8.5 for 240 months"
    else:
        raw_result = "To calculate your loan EMI, please provide principal, annual interest rate, and tenure in months. Example: EMI for 500000 at 8.5 for 240 months"
    return {**state, "context": "", "answer": raw_result}

def answer_node(state: FinState) -> FinState:
    llm              = get_llm()
    lang             = state.get("language", "en")
    lang_instruction = LANGUAGE_INSTRUCTIONS[lang]
    name_note        = f"The user's name is {state['user_name']}. " if state.get("user_name") else ""
    history          = "\n".join(f"{m['role']}: {m['content']}" for m in state["messages"][-6:])
    context          = state.get("context", "")
    raw_ans          = state.get("answer", "")
    retry_note       = " IMPORTANT: Your previous answer was not faithful — be more precise and stay within the context." if state.get("eval_retries", 0) > 0 else ""

    if raw_ans and not context:
        prompt = (
            f"{lang_instruction}\n\n"
            f"{name_note}Present the following loan EMI calculation result to the user in a clear, friendly way.\n"
            "Translate all labels and explanatory text into the required language.\n"
            "Keep all numbers exactly as they are.\n\n"
            f"Result:\n{raw_ans}\n\n"
            f"REMINDER: {lang_instruction}"
        )
    elif context:
        prompt = (
            f"{lang_instruction}\n\n"
            f"{name_note}You are FinAdvisor, a personal finance assistant for users.{retry_note}\n\n"
            "RULES:\n"
            "- Answer ONLY using the context below.\n"
            "- If the answer is not in the context, say so clearly in the required language.\n"
            "- Do NOT mix languages.\n\n"
            f"Context:\n{context}\n\n"
            f"Conversation history:\n{history}\n\n"
            f"Question: {state['question']}\n\n"
            f"REMINDER: {lang_instruction}"
        )
    else:
        prompt = (
            f"{lang_instruction}\n\n"
            f"{name_note}You are FinAdvisor, a helpful personal finance assistant for users.\n"
            "Respond warmly and briefly. Let the user know you can help with budgeting, SIP, tax saving,\n"
            "loans, insurance, stocks, retirement planning, and debt management.\n\n"
            f"Question: {state['question']}\n\n"
            f"REMINDER: {lang_instruction}"
        )
    answer = llm.invoke(prompt).content.strip()
    return {**state, "answer": answer}

def eval_node(state: FinState) -> FinState:
    if not state.get("context"):
        return {**state, "faithfulness": 1.0}
    llm    = get_llm()
    prompt = (
        "Score the faithfulness of the answer to the context on a scale of 0.0 to 1.0.\n"
        "1.0 = answer uses only information from the context. 0.0 = completely made up.\n\n"
        f"Context:\n{state['context'][:800]}\n\n"
        f"Answer:\n{state['answer'][:500]}\n\n"
        "Reply with ONLY a decimal number between 0.0 and 1.0."
    )
    try:
        score = float(llm.invoke(prompt).content.strip())
        score = max(0.0, min(1.0, score))
    except Exception:
        score = 0.8
    retries = state.get("eval_retries", 0) + 1
    return {**state, "faithfulness": score, "eval_retries": retries}

def save_node(state: FinState) -> FinState:
    msgs = state.get("messages", [])
    msgs = (msgs + [{"role": "assistant", "content": state["answer"]}])[-12:]
    return {**state, "messages": msgs}

def route_decision(state: FinState) -> str:
    return state["route"]

def eval_decision(state: FinState) -> str:
    if state.get("faithfulness", 1.0) < 0.7 and state.get("eval_retries", 0) < 2:
        return "answer"
    return "save"

@st.cache_resource(show_spinner="Building agent graph...")
def build_graph():
    memory = MemorySaver()
    g = StateGraph(FinState)
    g.add_node("memory",    memory_node)
    g.add_node("router",    router_node)
    g.add_node("retrieval", retrieval_node)
    g.add_node("skip",      skip_node)
    g.add_node("tool",      tool_node)
    g.add_node("answer",    answer_node)
    g.add_node("eval",      eval_node)
    g.add_node("save",      save_node)
    g.set_entry_point("memory")
    g.add_edge("memory", "router")
    g.add_conditional_edges("router", route_decision, {
        "retrieve": "retrieval",
        "chitchat": "skip",
        "tool":     "tool",
    })
    g.add_edge("retrieval", "answer")
    g.add_edge("skip",      "answer")
    g.add_edge("tool",      "answer")
    g.add_edge("answer",    "eval")
    g.add_conditional_edges("eval", eval_decision, {
        "answer": "answer",
        "save":   "save",
    })
    g.add_edge("save", END)
    return g.compile(checkpointer=memory)

st.set_page_config(page_title="FinAdvisor", page_icon=None, layout="centered")

graph    = build_graph()
vectordb = load_vectordb()

if "thread_id" not in st.session_state:
    st.session_state.thread_id = str(uuid.uuid4())
if "messages" not in st.session_state:
    st.session_state.messages = []
if "fin_state" not in st.session_state:
    st.session_state.fin_state = {
        "question": "", "messages": [], "route": "", "context": "",
        "answer": "", "language": "en", "faithfulness": 1.0,
        "eval_retries": 0, "user_name": None,
    }

with st.sidebar:
    st.title("Settings")
    language = st.selectbox(
        "Language",
        options=["en", "hi", "bn"],
        format_func=lambda x: {"en": "English", "hi": "Hindi", "bn": "Bengali"}[x],
    )
    if language != st.session_state.fin_state.get("language"):
        st.session_state.fin_state["language"] = language
    st.markdown("---")
    if st.button("New Conversation"):
        st.session_state.messages  = []
        st.session_state.thread_id = str(uuid.uuid4())
        st.session_state.fin_state = {
            "question": "", "messages": [], "route": "", "context": "",
            "answer": "", "language": language, "faithfulness": 1.0,
            "eval_retries": 0, "user_name": None,
        }
        st.rerun()
    st.markdown("---")
    st.caption(
        "General financial information only. "
        "Consult a SEBI-registered advisor for major decisions."
    )

st.title("FinAdvisor")
st.caption("Multilingual Personal Finance Assistant — English, Hindi, Bengali")
st.divider()

for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        st.markdown(msg["content"])

user_input = st.chat_input("Ask about budgeting, SIP, loans, tax saving, insurance...")
if user_input:
    st.session_state.messages.append({"role": "user", "content": user_input})
    with st.chat_message("user"):
        st.markdown(user_input)
    with st.chat_message("assistant"):
        with st.spinner("Thinking..."):
            state = st.session_state.fin_state.copy()
            state["question"] = user_input
            state["language"] = language
            try:
                config = {"configurable": {"thread_id": st.session_state.thread_id}}
                result = graph.invoke(state, config)
                answer = result.get("answer", "Something went wrong. Please try again.")
                st.session_state.fin_state = result
            except Exception as e:
                answer = f"Error: {e}"
        st.markdown(answer)
    st.session_state.messages.append({"role": "assistant", "content": answer})

'''

with open("app.py", "w", encoding="utf-8") as f:
    f.write(APP_CODE)
print("app.py written.\n")

In [ ]:
from pyngrok import ngrok, conf

if NGROK_TOKEN:
    conf.get_default().auth_token = NGROK_TOKEN
    print("ngrok token set.")
else:
    print("WARNING: NGROK_AUTH_TOKEN not found. Add it to Colab secrets.")

In [ ]:
import subprocess, time
from pyngrok import ngrok

ngrok.kill()
subprocess.run(["pkill", "-f", "streamlit"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
subprocess.run(["pkill", "ngrok"],           stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(2)

subprocess.Popen(
    ["streamlit", "run", "app.py",
     "--server.port", "8501",
     "--server.enableCORS", "false",
     "--server.enableXsrfProtection", "false"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
time.sleep(5)

public_url = ngrok.connect(8501)
print("\nFinAdvisor is live at:\n")
print(public_url.public_url)

In [ ]:
# Run this cell to stop the tunnel when done
!pkill ngrok